In [1]:
# [CELL 1] 📦 IMPORTS
# ======================================================================
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports loaded!")


✅ Imports loaded!


In [2]:
# [CELL 2] 📁 DATA LOADING
# ======================================================================
DATA_FOLDER = '../Experiment_Code/DATA'
STIMULI_SCALAR = 6.5  # From views.py

print("=" * 80)
print("📁 LOADING DATA")
print("=" * 80)

# Connect to database - Check both locations and use the newer one
import os
from datetime import datetime

primary_db_path = f'{DATA_FOLDER}/db.sqlite3'
alt_path = f'{DATA_FOLDER}/pilot_20251215/db.sqlite3'

# Check which database exists and is newer
db_candidates = []
if os.path.exists(primary_db_path):
    mtime = os.path.getmtime(primary_db_path)
    db_candidates.append((primary_db_path, mtime))
if os.path.exists(alt_path):
    mtime = os.path.getmtime(alt_path)
    db_candidates.append((alt_path, mtime))

if len(db_candidates) == 0:
    raise FileNotFoundError(f"Database not found at {primary_db_path} or {alt_path}\n"
                           f"Please ensure db.sqlite3 exists in {DATA_FOLDER}")

# Use the newest database
db_candidates.sort(key=lambda x: x[1], reverse=True)  # Sort by modification time, newest first
primary_db_path = db_candidates[0][0]

if len(db_candidates) > 1:
    print(f"Found {len(db_candidates)} database files:")
    for path, mtime in db_candidates:
        mod_time = datetime.fromtimestamp(mtime)
        print(f"  - {path} (modified: {mod_time})")
    print(f"Using: {primary_db_path} (newest)")
else:
    print(f"Using database: {primary_db_path}")

# Check if database has tables
test_conn = sqlite3.connect(primary_db_path)
cursor = test_conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]
test_conn.close()

if len(tables) == 0:
    print(f"⚠️  Database file exists but is EMPTY (no tables): {primary_db_path}")
    print(f"   This means the database hasn't been populated with data yet.")
    print(f"   Options:")
    print(f"   1. Export data from PythonAnywhere to this location")
    print(f"   2. Check if data is in a subfolder (e.g., pilot_20251215/)")
    
    # Check if there's data in subfolder
    alt_path = f'{DATA_FOLDER}/pilot_20251215/db.sqlite3'
    if os.path.exists(alt_path):
        test_conn = sqlite3.connect(alt_path)
        cursor = test_conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        alt_tables = [row[0] for row in cursor.fetchall()]
        test_conn.close()
        if len(alt_tables) > 0 and 'experiment_experimentdata' in alt_tables:
            print(f"   ✅ Found database with data at: {alt_path}")
            primary_db_path = alt_path
            tables = alt_tables
        else:
            raise ValueError(f"Database at {primary_db_path} is empty and alternative at {alt_path} also has no data.")
    else:
        raise ValueError(f"Database at {primary_db_path} is empty. Please export data from PythonAnywhere first.")
elif 'experiment_experimentdata' not in tables:
    print(f"⚠️  Database has tables but missing 'experiment_experimentdata':")
    print(f"   Available tables: {tables}")
    raise ValueError(f"Table 'experiment_experimentdata' not found. Available tables: {tables}")
else:
    print(f"✅ Database found with tables: {primary_db_path}")

# Connect to the verified database
conn = sqlite3.connect(primary_db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]
print(f"✅ Using database: {primary_db_path}")
print(f"✅ Tables: {tables}")

# Load users
try:
    users_df = pd.read_sql_query("""
        SELECT user_id, aid, csv_row_id, ps, human_sensitivity, ds_sensitivity,
               start_time, complete, end_time
        FROM experiment_experimentdata
    """, conn)
    print("✅ Loaded users with csv_row_id")
except Exception as e:
    if 'csv_row_id' in str(e):
        users_df = pd.read_sql_query("""
            SELECT user_id, aid, ps, human_sensitivity, ds_sensitivity,
                   start_time, complete, end_time
            FROM experiment_experimentdata
        """, conn)
        users_df['csv_row_id'] = None
        print("⚠️  csv_row_id column not found")
    else:
        raise

# Load actions
actions_df = pd.read_sql_query("""
    SELECT ea.user_id_id as user_id, ea.block_number, ea.trial_number,
           ea.classification_decision, ea.stimulus_seen, ea.dss_judgment,
           ea.decision_time, ea.correct_classification
    FROM experiment_experimentaction ea
""", conn)

# Load TOAST
toast_df = pd.read_sql_query("""
    SELECT tr.user_id_id as user_id, 
           tr.usefulness, tr.reliability, tr.trust, tr.confidence,
           tr.satisfaction
    FROM experiment_toastresponse tr
""", conn)

conn.close()

# Load conditions CSV - Check both locations and use the newer one
possible_csv_paths = [
    f'{DATA_FOLDER}/conditions_experiment_3ps_11x11_120_A.csv',
    f'{DATA_FOLDER}/pilot_20251215/conditions_experiment_3ps_11x11_120_A.csv',
    '../Experiment_Code/DATA/pilot_20251215/conditions_experiment_3ps_11x11_120_A.csv',
    '../Experiment_Code/DATA/conditions_experiment_3ps_11x11_120_A.csv'
]

# Find all existing CSV files and use the newest one
csv_candidates = []
for path in possible_csv_paths:
    if os.path.exists(path):
        mtime = os.path.getmtime(path)
        csv_candidates.append((path, mtime))

if len(csv_candidates) == 0:
    print("\n❌ Conditions CSV not found. Searched in:")
    for path in possible_csv_paths:
        exists = os.path.exists(path)
        print(f"   {'✅' if exists else '❌'} {path}")
    raise FileNotFoundError(f"\nConditions CSV not found.\nPlease update DATA_FOLDER or ensure the CSV exists.")

# Use the newest CSV
csv_candidates.sort(key=lambda x: x[1], reverse=True)  # Sort by modification time, newest first
conditions_path = csv_candidates[0][0]

if len(csv_candidates) > 1:
    print(f"\nFound {len(csv_candidates)} CSV files:")
    for path, mtime in csv_candidates:
        mod_time = datetime.fromtimestamp(mtime)
        print(f"  - {path} (modified: {mod_time})")
    print(f"Using: {conditions_path} (newest)")

conditions_df = pd.read_csv(conditions_path)
print(f"✅ Conditions CSV loaded: {conditions_path}")

# Date filter (Dec 14, 2025 onwards)
MIN_DATE = '2025-12-14'
users_df['start_date'] = pd.to_datetime(users_df['start_time']).dt.date
users_before = len(users_df)
users_df = users_df[users_df['start_date'] >= pd.to_datetime(MIN_DATE).date()]
users_filtered = users_before - len(users_df)
print(f"📅 Date filter: Removed {users_filtered} users from before {MIN_DATE}")

# Filter related data
valid_user_ids = users_df['user_id'].tolist()
actions_df = actions_df[actions_df['user_id'].isin(valid_user_ids)]
toast_df = toast_df[toast_df['user_id'].isin(valid_user_ids)]

print(f"\n✅ Data loaded:")
print(f"   Users: {len(users_df)} (Complete: {users_df['complete'].sum()})")
print(f"   Actions: {len(actions_df)} trials")
print(f"   TOAST: {len(toast_df)} responses")
print(f"   Conditions: {len(conditions_df)} rows")


📁 LOADING DATA
Found 2 database files:
  - ../Experiment_Code/DATA/pilot_20251215/db.sqlite3 (modified: 2025-12-17 00:40:26.496853)
  - ../Experiment_Code/DATA/db.sqlite3 (modified: 2025-12-16 23:48:40.831235)
Using: ../Experiment_Code/DATA/pilot_20251215/db.sqlite3 (newest)
✅ Database found with tables: ../Experiment_Code/DATA/pilot_20251215/db.sqlite3
✅ Using database: ../Experiment_Code/DATA/pilot_20251215/db.sqlite3
✅ Tables: ['django_migrations', 'sqlite_sequence', 'auth_group_permissions', 'auth_user_groups', 'auth_user_user_permissions', 'django_admin_log', 'django_content_type', 'auth_permission', 'auth_group', 'auth_user', 'experiment_experimentaction', 'experiment_toastresponse', 'experiment_experimentdata', 'django_session']
✅ Loaded users with csv_row_id

Found 4 CSV files:
  - ../Experiment_Code/DATA/pilot_20251215/conditions_experiment_3ps_11x11_120_A.csv (modified: 2025-12-17 00:40:21.931660)
  - ../Experiment_Code/DATA/pilot_20251215/conditions_experiment_3ps_11x11_120_

In [3]:
# [CELL 3] ✅ BASIC DATA INTEGRITY CHECKS
# ======================================================================
print("\n" + "=" * 80)
print("✅ BASIC DATA INTEGRITY CHECKS")
print("=" * 80)

issues = []

# Check 1: csv_row_id presence
users_with_row = users_df[users_df['csv_row_id'].notna()]
users_without_row = users_df[users_df['csv_row_id'].isna()]
print(f"\n1. csv_row_id assignment:")
print(f"   ✅ Users with csv_row_id: {len(users_with_row)}")
if len(users_without_row) > 0:
    print(f"   ⚠️  Users without csv_row_id: {len(users_without_row)}")
    issues.append(f"{len(users_without_row)} users without csv_row_id")

# Check 2: Duplicate csv_row_id for completed users
completed_users = users_df[users_df['complete'] == True]
if len(completed_users) > 0 and 'csv_row_id' in completed_users.columns:
    row_counts = completed_users['csv_row_id'].value_counts()
    duplicate_rows = row_counts[row_counts > 1]
    print(f"\n2. Duplicate csv_row_id (completed users):")
    if len(duplicate_rows) > 0:
        print(f"   ⚠️  {len(duplicate_rows)} CSV rows assigned to multiple completed users (expected from race condition)")
        for row_id, count in duplicate_rows.items():
            user_ids = completed_users[completed_users['csv_row_id'] == row_id]['user_id'].tolist()
            print(f"      Row {row_id}: {count} users {user_ids}")
        # Don't add to issues - this is expected from pre-fix race condition
    else:
        print(f"   ✅ Each completed user has unique csv_row_id")

# Check 3: CSV used flags
print(f"\n3. CSV used flags:")
used_0 = len(conditions_df[conditions_df['used'] == 0])
used_05 = len(conditions_df[conditions_df['used'] == 0.5])
used_1 = len(conditions_df[conditions_df['used'] == 1])
print(f"   Used=0 (Available): {used_0}")
print(f"   Used=0.5 (In-progress): {used_05}")
print(f"   Used=1 (Completed): {used_1}")

# Check 4: Users vs CSV used flags
print(f"\n4. Users vs CSV used flags:")
# Ignore incomplete users on rows shared with completed users (race condition)

if len(users_with_row) > 0:
    assigned_rows = users_with_row['csv_row_id'].unique()
    flag_mismatches = []
    shared_rows_ignored = []
    
    for row_id in assigned_rows:
        csv_row = conditions_df[conditions_df['id'] == row_id]
        if len(csv_row) > 0:
            csv_used = csv_row.iloc[0]['used']
            users_on_row = users_with_row[users_with_row['csv_row_id'] == row_id]
            complete_count = users_on_row['complete'].sum()
            incomplete_count = len(users_on_row) - complete_count
            
            # If row has both completed and incomplete users, it's a shared row from race condition
            # In this case, used=1 is correct (because completed user exists), ignore incomplete users
            if complete_count > 0 and incomplete_count > 0:
                if csv_used == 1:
                    shared_rows_ignored.append(row_id)  # Correctly marked, just ignore
                else:
                    flag_mismatches.append(row_id)  # Should be 1 but isn't
            elif complete_count > 0 and csv_used != 1:
                # Row has only completed users but not marked used=1
                flag_mismatches.append(row_id)
            elif complete_count == 0 and csv_used == 1:
                # Row marked used=1 but has no completed users (and no incomplete either)
                # Check if it has any users at all
                if len(users_on_row) == 0:
                    flag_mismatches.append(row_id)  # Marked 1 but no users
    
    if len(shared_rows_ignored) > 0:
        print(f"   ℹ️  {len(shared_rows_ignored)} rows with both completed and incomplete users (race condition - ignored)")
    
    if len(flag_mismatches) > 0:
        print(f"   ⚠️  {len(flag_mismatches)} CSV rows with flag mismatches (cosmetic - CSV flags not updated, but data is correct)")
        print(f"      Example rows: {flag_mismatches[:5]}")
        # Don't add to issues - this is cosmetic, data integrity is fine
    else:
        print(f"   ✅ All CSV flags match user completion status")

if len(issues) == 0:
    print("\n✅ All basic integrity checks passed!")
else:
    print(f"\n⚠️  Found {len(issues)} issue(s)")



✅ BASIC DATA INTEGRITY CHECKS

1. csv_row_id assignment:
   ✅ Users with csv_row_id: 93

2. Duplicate csv_row_id (completed users):
   ⚠️  3 CSV rows assigned to multiple completed users (expected from race condition)
      Row 304: 2 users [50, 51]
      Row 65: 2 users [91, 93]
      Row 155: 2 users [36, 73]

3. CSV used flags:
   Used=0 (Available): 295
   Used=0.5 (In-progress): 2
   Used=1 (Completed): 66

4. Users vs CSV used flags:
   ℹ️  4 rows with both completed and incomplete users (race condition - ignored)
   ✅ All CSV flags match user completion status

✅ All basic integrity checks passed!


In [4]:
# [CELL 5] 🎯 DS DECISION VERIFICATION
# ======================================================================
# Verify DS decisions match CSV (threshold > 0 for s_t columns)
print("\n" + "=" * 80)
print("🎯 DS DECISION VERIFICATION")
print("=" * 80)

def verify_ds_decision(csv_row, trial_num, block_num):
    """Verify DS decision matches CSV threshold logic (>0)"""
    if block_num == 1:
        csv_col = f's_t{trial_num:02d}'
        if csv_col not in csv_row:
            return None, "Column not found"
        s_t = float(csv_row[csv_col])
        expected_ds = 1 if s_t > 0 else 0
        csv_ds = int(csv_row.get(f'ds_dec_t{trial_num:02d}', -1))
        return expected_ds == csv_ds, f"s_t={s_t:.2f}, expected={expected_ds}, CSV={csv_ds}"
    elif block_num == 2:
        csv_t = trial_num + 10
        csv_col = f's_t{csv_t:02d}'
        if csv_col not in csv_row:
            return None, "Column not found"
        s_t = float(csv_row[csv_col])
        expected_ds = 1 if s_t > 0 else 0
        csv_ds = int(csv_row.get(f'ds_dec_t{csv_t:02d}', -1))
        return expected_ds == csv_ds, f"s_t={s_t:.2f}, expected={expected_ds}, CSV={csv_ds}"
    else:  # block_num == 3
        csv_t = trial_num + 20
        csv_col = f's_t{csv_t:02d}'
        if csv_col not in csv_row:
            return None, "Column not found"
        s_t = float(csv_row[csv_col])
        expected_ds = 1 if s_t > 0 else 0
        csv_ds = int(csv_row.get(f'ds_dec_t{csv_t:02d}', -1))
        return expected_ds == csv_ds, f"s_t={s_t:.2f}, expected={expected_ds}, CSV={csv_ds}"

# Verify DS decisions for sample users
ds_errors = []
sample_size = min(5, len(completed_users))

for _, user in completed_users.head(sample_size).iterrows():
    if pd.isna(user['csv_row_id']):
        continue
    
    csv_row_id = int(user['csv_row_id'])
    csv_row = conditions_df[conditions_df['id'] == csv_row_id].iloc[0]
    user_actions = actions_df[actions_df['user_id'] == user['user_id']].sort_values(['block_number', 'trial_number'])
    
    user_errors = []
    for _, action in user_actions.head(20).iterrows():  # Check first 20 trials
        block = int(action['block_number'])
        trial = int(action['trial_number'])
        # Normalize DS judgment (handle both string and int formats)
        ds_judgment_raw = action['dss_judgment']
        if pd.isna(ds_judgment_raw):
            db_ds = -1
        elif isinstance(ds_judgment_raw, str):
            db_ds = 1 if ds_judgment_raw.lower() == 'signal' else 0
        else:
            db_ds = int(ds_judgment_raw)
        
        is_correct, msg = verify_ds_decision(csv_row, trial, block)
        if is_correct is False:
            user_errors.append(f"B{block}T{trial}: {msg}, DB_DS={db_ds}")
    
    if len(user_errors) > 0:
        print(f"\n❌ User {user['user_id']} (Row {csv_row_id}): {len(user_errors)} DS decision errors:")
        for err in user_errors[:5]:  # Show first 5
            print(f"   {err}")
        ds_errors.extend(user_errors)

if len(ds_errors) == 0:
    print("\n✅ All DS decisions verified correctly!")
else:
    print(f"\n⚠️  Found {len(ds_errors)} DS decision error(s)")



🎯 DS DECISION VERIFICATION

✅ All DS decisions verified correctly!


In [5]:
# [CELL 6] 📊 STIMULUS & EVENT TYPE MATCHING
# ======================================================================
# Verify stimulus_seen matches CSV (h_t + STIMULI_SCALAR) and event matches
print("\n" + "=" * 80)
print("📊 STIMULUS & EVENT TYPE MATCHING")
print("=" * 80)

def get_csv_stimulus(csv_row, trial_num, block_num):
    """Get expected stimulus from CSV"""
    if block_num == 1:
        csv_col = f'h_t{trial_num:02d}'
        return float(csv_row[csv_col]) + STIMULI_SCALAR
    elif block_num == 2:
        csv_t = trial_num + 10
        csv_col = f'h_t{csv_t:02d}'
        return float(csv_row[csv_col]) + STIMULI_SCALAR
    else:  # block_num == 3
        csv_t = trial_num + 20
        csv_col = f'h_t{csv_t:02d}'
        return float(csv_row[csv_col]) + STIMULI_SCALAR

def get_csv_event(csv_row, trial_num, block_num):
    """Get expected event type from CSV"""
    if block_num == 1:
        csv_col = f'event_t{trial_num:02d}'
        return csv_row[csv_col]
    elif block_num == 2:
        csv_t = trial_num + 10
        csv_col = f'event_t{csv_t:02d}'
        return csv_row[csv_col]
    else:  # block_num == 3
        csv_t = trial_num + 20
        csv_col = f'event_t{csv_t:02d}'
        return csv_row[csv_col]

stimulus_errors = []
event_errors = []

sample_size = min(5, len(completed_users))
for _, user in completed_users.head(sample_size).iterrows():
    if pd.isna(user['csv_row_id']):
        continue
    
    csv_row_id = int(user['csv_row_id'])
    csv_row = conditions_df[conditions_df['id'] == csv_row_id].iloc[0]
    user_actions = actions_df[actions_df['user_id'] == user['user_id']].sort_values(['block_number', 'trial_number'])
    
    for _, action in user_actions.head(20).iterrows():
        block = int(action['block_number'])
        trial = int(action['trial_number'])
        
        # Check stimulus
        expected_stim = get_csv_stimulus(csv_row, trial, block)
        actual_stim = float(action['stimulus_seen'])
        if abs(expected_stim - actual_stim) > 0.01:
            stimulus_errors.append(f"User {user['user_id']} B{block}T{trial}: expected={expected_stim:.2f}, actual={actual_stim:.2f}")
        
        # Check event type
        expected_event = get_csv_event(csv_row, trial, block)
        actual_event = action['correct_classification']
        # Handle both string and int formats
        if isinstance(expected_event, (int, float)):
            expected_event = 'signal' if expected_event == 1 else 'noise'
        if expected_event.lower() != actual_event.lower():
            event_errors.append(f"User {user['user_id']} B{block}T{trial}: expected={expected_event}, actual={actual_event}")

print(f"\nStimulus matching: {len(stimulus_errors)} error(s)")
if len(stimulus_errors) > 0:
    for err in stimulus_errors[:5]:
        print(f"   {err}")

print(f"\nEvent type matching: {len(event_errors)} error(s)")
if len(event_errors) > 0:
    for err in event_errors[:5]:
        print(f"   {err}")

if len(stimulus_errors) == 0 and len(event_errors) == 0:
    print("\n✅ All stimulus and event types match!")
else:
    print(f"\n⚠️  Found {len(stimulus_errors) + len(event_errors)} mismatch(es)")



📊 STIMULUS & EVENT TYPE MATCHING

Stimulus matching: 0 error(s)

Event type matching: 0 error(s)

✅ All stimulus and event types match!


In [6]:
# [CELL 7] 🔢 BLOCK 3 COLUMN MAPPING VERIFICATION
# ======================================================================
# Critical: Block 3 trial 1 should use CSV column 21, not 1
print("\n" + "=" * 80)
print("🔢 BLOCK 3 COLUMN MAPPING VERIFICATION")
print("=" * 80)

block3_errors = []

for _, user in completed_users.head(10).iterrows():
    if pd.isna(user['csv_row_id']):
        continue
    
    csv_row_id = int(user['csv_row_id'])
    csv_row = conditions_df[conditions_df['id'] == csv_row_id].iloc[0]
    
    # Check Block 3 trial 1
    b3t1 = actions_df[(actions_df['user_id'] == user['user_id']) & 
                      (actions_df['block_number'] == 3) & 
                      (actions_df['trial_number'] == 1)]
    
    if len(b3t1) > 0:
        actual_event = b3t1.iloc[0]['correct_classification']
        csv_event_21 = csv_row['event_t21']
        csv_event_01 = csv_row.get('event_t01', None)
        
        # Normalize event values
        if isinstance(csv_event_21, (int, float)):
            csv_event_21 = 'signal' if csv_event_21 == 1 else 'noise'
        if csv_event_01 and isinstance(csv_event_01, (int, float)):
            csv_event_01 = 'signal' if csv_event_01 == 1 else 'noise'
        
        if actual_event.lower() == csv_event_21.lower():
            print(f"✅ User {user['user_id']}: Block 3 trial 1 correctly uses column 21")
        elif csv_event_01 and actual_event.lower() == csv_event_01.lower():
            print(f"❌ User {user['user_id']}: Block 3 trial 1 uses column 1 instead of 21!")
            block3_errors.append(f"User {user['user_id']}: B3T1 uses column 1")
        else:
            print(f"⚠️  User {user['user_id']}: Block 3 mapping unclear")
            block3_errors.append(f"User {user['user_id']}: B3T1 mapping unclear")

if len(block3_errors) == 0:
    print("\n✅ Block 3 column mapping is correct!")
else:
    print(f"\n⚠️  Found {len(block3_errors)} Block 3 mapping error(s)")



🔢 BLOCK 3 COLUMN MAPPING VERIFICATION
✅ User 5: Block 3 trial 1 correctly uses column 21
✅ User 6: Block 3 trial 1 correctly uses column 21
✅ User 8: Block 3 trial 1 correctly uses column 21
✅ User 9: Block 3 trial 1 correctly uses column 21
✅ User 10: Block 3 trial 1 correctly uses column 21
✅ User 11: Block 3 trial 1 correctly uses column 21
✅ User 12: Block 3 trial 1 correctly uses column 21
✅ User 13: Block 3 trial 1 correctly uses column 21
✅ User 14: Block 3 trial 1 correctly uses column 21
✅ User 15: Block 3 trial 1 correctly uses column 21

✅ Block 3 column mapping is correct!


In [7]:
# [CELL 8] 📈 TRIAL SEQUENCE VALIDATION
# ======================================================================
# Verify trial sequences are correct (no gaps, correct numbering)
print("\n" + "=" * 80)
print("📈 TRIAL SEQUENCE VALIDATION")
print("=" * 80)

sequence_issues = []

for _, user in completed_users.head(10).iterrows():
    user_actions = actions_df[actions_df['user_id'] == user['user_id']].sort_values(['block_number', 'trial_number'])
    
    # Check each block
    for block in [1, 2, 3]:
        block_actions = user_actions[user_actions['block_number'] == block]
        if len(block_actions) == 0:
            continue
        
        expected_trials = list(range(1, len(block_actions) + 1))
        actual_trials = sorted(block_actions['trial_number'].unique().tolist())
        
        if expected_trials != actual_trials:
            missing = set(expected_trials) - set(actual_trials)
            extra = set(actual_trials) - set(expected_trials)
            if missing or extra:
                sequence_issues.append(f"User {user['user_id']} Block {block}: missing={missing}, extra={extra}")

if len(sequence_issues) == 0:
    print("✅ All trial sequences are correct!")
else:
    print(f"⚠️  Found {len(sequence_issues)} sequence issue(s):")
    for issue in sequence_issues[:5]:
        print(f"   {issue}")



📈 TRIAL SEQUENCE VALIDATION
✅ All trial sequences are correct!


In [8]:
# [CELL 9] 📊 PERFORMANCE METRICS VALIDATION
# ======================================================================
# Calculate and validate performance metrics (from ML analysis checks)
print("\n" + "=" * 80)
print("📊 PERFORMANCE METRICS VALIDATION")
print("=" * 80)

# Calculate confusion matrix metrics per user
def calc_user_metrics(user_id, actions_subset):
    """Calculate performance metrics for a user"""
    TP = len(actions_subset[(actions_subset['correct_classification'] == 'signal') & 
                            (actions_subset['classification_decision'] == 'signal')])
    TN = len(actions_subset[(actions_subset['correct_classification'] == 'noise') & 
                            (actions_subset['classification_decision'] == 'noise')])
    FP = len(actions_subset[(actions_subset['correct_classification'] == 'noise') & 
                            (actions_subset['classification_decision'] == 'signal')])
    FN = len(actions_subset[(actions_subset['correct_classification'] == 'signal') & 
                            (actions_subset['classification_decision'] == 'noise')])
    
    total = TP + TN + FP + FN
    if total == 0:
        return None
    
    accuracy = (TP + TN) / total
    hit_rate = TP / (TP + FN) if (TP + FN) > 0 else 0
    fa_rate = FP / (FP + TN) if (FP + TN) > 0 else 0
    
    # DS agreement (for blocks with DS)
    ds_actions = actions_subset[actions_subset['dss_judgment'].notna()]
    if len(ds_actions) > 0:
        # Handle both string and int formats for DS judgment
        def normalize_ds_judgment(val):
            if pd.isna(val):
                return None
            if isinstance(val, str):
                return 1 if val.lower() == 'signal' else 0
            return int(val)
        
        ds_actions_normalized = ds_actions.copy()
        ds_actions_normalized['dss_judgment_norm'] = ds_actions_normalized['dss_judgment'].apply(normalize_ds_judgment)
        
        agreed = ((ds_actions_normalized['classification_decision'] == 'signal') & (ds_actions_normalized['dss_judgment_norm'] == 1)) | \
                 ((ds_actions_normalized['classification_decision'] == 'noise') & (ds_actions_normalized['dss_judgment_norm'] == 0))
        ds_agreement = agreed.sum() / len(ds_actions)
    else:
        ds_agreement = None
    
    return {
        'user_id': user_id,
        'total_trials': total,
        'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
        'accuracy': accuracy,
        'hit_rate': hit_rate,
        'fa_rate': fa_rate,
        'ds_agreement': ds_agreement
    }

# Calculate metrics for all complete users
user_metrics = []
for _, user in completed_users.iterrows():
    user_actions = actions_df[actions_df['user_id'] == user['user_id']]
    metrics = calc_user_metrics(user['user_id'], user_actions)
    if metrics:
        user_metrics.append(metrics)

metrics_df = pd.DataFrame(user_metrics)

print(f"\nCalculated metrics for {len(metrics_df)} users:")
print(f"   Average accuracy: {metrics_df['accuracy'].mean():.3f}")
print(f"   Average hit rate: {metrics_df['hit_rate'].mean():.3f}")
print(f"   Average FA rate: {metrics_df['fa_rate'].mean():.3f}")
if 'ds_agreement' in metrics_df.columns:
    ds_agreement_mean = metrics_df['ds_agreement'].dropna().mean()
    print(f"   Average DS agreement: {ds_agreement_mean:.3f}")

# Check for suspicious patterns
suspicious = []
if len(metrics_df) > 0:
    # Very high accuracy (>95%)
    very_high_acc = metrics_df[metrics_df['accuracy'] > 0.95]
    if len(very_high_acc) > 0:
        print(f"\n⚠️  {len(very_high_acc)} user(s) with very high accuracy (>95%):")
        print(very_high_acc[['user_id', 'accuracy', 'total_trials']].to_string(index=False))
    
    # Very low accuracy (<50%)
    very_low_acc = metrics_df[metrics_df['accuracy'] < 0.50]
    if len(very_low_acc) > 0:
        print(f"\n⚠️  {len(very_low_acc)} user(s) with very low accuracy (<50%):")
        print(very_low_acc[['user_id', 'accuracy', 'total_trials']].to_string(index=False))



📊 PERFORMANCE METRICS VALIDATION

Calculated metrics for 68 users:
   Average accuracy: 0.714
   Average hit rate: 0.596
   Average FA rate: 0.247
   Average DS agreement: 0.717

⚠️  1 user(s) with very low accuracy (<50%):
 user_id  accuracy  total_trials
      26      0.45           120


In [9]:
# [CELL 12] 🔬 STATISTICAL TESTS FOR EXPERIMENT SETUP
# ======================================================================
# Verify experimental design: balance, distributions, etc.
print("\n" + "=" * 80)
print("🔬 STATISTICAL TESTS FOR EXPERIMENT SETUP")
print("=" * 80)

from scipy import stats

# Test 1: Balance of ps levels
print("\n1. Balance of ps (signal probability) levels:")
ps_counts = completed_users['ps'].value_counts().sort_index()
print(ps_counts)
if len(ps_counts) == 3:
    # Expected: roughly equal distribution
    expected = len(completed_users) / 3
    chi2, p_val = stats.chisquare(ps_counts, f_exp=[expected]*3)
    print(f"   Chi-square test: χ²={chi2:.2f}, p={p_val:.4f}")
    if p_val > 0.05:
        print(f"   ✅ ps levels are balanced (p > 0.05)")
    else:
        print(f"   ⚠️  ps levels may not be balanced (p ≤ 0.05)")

# Test 2: Balance of d'_human levels
print("\n2. Balance of d'_human levels:")
d_h_counts = completed_users['human_sensitivity'].value_counts().sort_index()
print(f"   Unique d'_human values: {sorted(completed_users['human_sensitivity'].unique())}")
print(f"   Counts: {dict(d_h_counts)}")
if len(d_h_counts) >= 3:
    expected = len(completed_users) / len(d_h_counts)
    chi2, p_val = stats.chisquare(d_h_counts, f_exp=[expected]*len(d_h_counts))
    print(f"   Chi-square test: χ²={chi2:.2f}, p={p_val:.4f}")

# Test 3: Balance of d'_DS levels
print("\n3. Balance of d'_DS levels:")
d_s_counts = completed_users['ds_sensitivity'].value_counts().sort_index()
print(f"   Unique d'_DS values: {sorted(completed_users['ds_sensitivity'].unique())}")
print(f"   Counts: {dict(d_s_counts)}")
if len(d_s_counts) >= 3:
    expected = len(completed_users) / len(d_s_counts)
    chi2, p_val = stats.chisquare(d_s_counts, f_exp=[expected]*len(d_s_counts))
    print(f"   Chi-square test: χ²={chi2:.2f}, p={p_val:.4f}")

# Test 4: Independence of factors (ps, d'_h, d'_s)
print("\n4. Factor Independence Check:")
# Check if combinations are balanced
combinations = completed_users.groupby(['ps', 'human_sensitivity', 'ds_sensitivity']).size()
print(f"   Unique combinations: {len(combinations)}")
print(f"   Expected per combination (if balanced): {len(completed_users) / len(combinations):.1f}")
print(f"   Actual range: {combinations.min()} - {combinations.max()}")

# Test 5: Distribution of trial counts (should be 120 for complete users)
print("\n5. Trial Count Distribution:")
trial_counts = actions_df.groupby('user_id').size()
complete_trial_counts = trial_counts[trial_counts.index.isin(completed_users['user_id'])]
print(f"   Complete users - Mean trials: {complete_trial_counts.mean():.1f}")
print(f"   Complete users - Expected: 120")
print(f"   Complete users with 120 trials: {(complete_trial_counts == 120).sum()}/{len(complete_trial_counts)}")

if (complete_trial_counts == 120).sum() == len(complete_trial_counts):
    print(f"   ✅ All complete users have exactly 120 trials")
else:
    print(f"   ⚠️  Some complete users don't have 120 trials")

# Test 6: Reaction time distribution (should be reasonable)
print("\n6. Reaction Time (Decision Time) Distribution:")
rt_data = actions_df['decision_time'].dropna()
if len(rt_data) > 0:
    print(f"   Mean RT: {rt_data.mean():.2f}s")
    print(f"   Median RT: {rt_data.median():.2f}s")
    print(f"   Min RT: {rt_data.min():.2f}s")
    print(f"   Max RT: {rt_data.max():.2f}s")
    
    # Check for suspiciously fast responses (<0.1s might be random clicking)
    very_fast = (rt_data < 0.1).sum()
    if very_fast > 0:
        print(f"   ⚠️  {very_fast} responses with RT < 0.1s (possible random clicking)")
    
    # Check for suspiciously slow responses (>10s might be AFK)
    very_slow = (rt_data > 10).sum()
    if very_slow > 0:
        print(f"   ⚠️  {very_slow} responses with RT > 10s (possible AFK)")

print("\n✅ Statistical checks completed!")



🔬 STATISTICAL TESTS FOR EXPERIMENT SETUP

1. Balance of ps (signal probability) levels:
ps
0.20    21
0.35    21
0.50    26
Name: count, dtype: int64
   Chi-square test: χ²=0.74, p=0.6924
   ✅ ps levels are balanced (p > 0.05)

2. Balance of d'_human levels:
   Unique d'_human values: [np.float64(0.5), np.float64(0.7), np.float64(0.9), np.float64(1.1), np.float64(1.3), np.float64(1.5), np.float64(1.7), np.float64(1.9), np.float64(2.1), np.float64(2.3), np.float64(2.5)]
   Counts: {0.5: np.int64(5), 0.7: np.int64(7), 0.9: np.int64(4), 1.1: np.int64(6), 1.3: np.int64(11), 1.5: np.int64(9), 1.7: np.int64(7), 1.9: np.int64(4), 2.1: np.int64(3), 2.3: np.int64(5), 2.5: np.int64(7)}
   Chi-square test: χ²=9.00, p=0.5321

3. Balance of d'_DS levels:
   Unique d'_DS values: [np.float64(0.5), np.float64(0.7), np.float64(0.9), np.float64(1.1), np.float64(1.3), np.float64(1.5), np.float64(1.7), np.float64(1.9), np.float64(2.1), np.float64(2.3), np.float64(2.5)]
   Counts: {0.5: np.int64(6), 0.7: 

In [10]:
# [CELL 10] 📋 COMPREHENSIVE SUMMARY & REPORT
# ======================================================================
print("\n" + "=" * 80)
print("📋 COMPREHENSIVE VALIDATION SUMMARY")
print("=" * 80)

print(f"\n📊 DATA OVERVIEW:")
print(f"   Total users: {len(users_df)}")
print(f"   Complete users: {len(completed_users)}")
print(f"   Total actions: {len(actions_df)}")
print(f"   TOAST responses: {len(toast_df)}")
print(f"   CSV conditions: {len(conditions_df)} rows")

print(f"\n✅ VALIDATION CHECKS:")
print(f"   1. csv_row_id assignment: {'✅' if len(users_without_row) == 0 else '⚠️'}")
dup_check = len(completed_users['csv_row_id'].value_counts()[completed_users['csv_row_id'].value_counts() > 1]) if len(completed_users) > 0 and 'csv_row_id' in completed_users.columns else 0
print(f"   2. Duplicate csv_row_id: {'✅' if dup_check == 0 else '⚠️'}")
print(f"   3. CSV parameter matching: {'✅' if len(mismatches) == 0 else '⚠️'}")
print(f"   4. DS decision verification: {'✅' if len(ds_errors) == 0 else '⚠️'}")
print(f"   5. Stimulus matching: {'✅' if len(stimulus_errors) == 0 else '⚠️'}")
print(f"   6. Event type matching: {'✅' if len(event_errors) == 0 else '⚠️'}")
print(f"   7. Block 3 column mapping: {'✅' if len(block3_errors) == 0 else '⚠️'}")
print(f"   8. Trial sequence: {'✅' if len(sequence_issues) == 0 else '⚠️'}")

total_issues = (len(users_without_row) + dup_check + len(mismatches) + len(ds_errors) + 
               len(stimulus_errors) + len(event_errors) + len(block3_errors) + len(sequence_issues))

print(f"\n{'=' * 80}")
if total_issues == 0:
    print("✅ ALL VALIDATION CHECKS PASSED!")
else:
    print(f"⚠️  FOUND {total_issues} TOTAL ISSUE(S) - REVIEW ABOVE")
print(f"{'=' * 80}")



📋 COMPREHENSIVE VALIDATION SUMMARY

📊 DATA OVERVIEW:
   Total users: 93
   Complete users: 68
   Total actions: 8755
   TOAST responses: 71
   CSV conditions: 363 rows

✅ VALIDATION CHECKS:
   1. csv_row_id assignment: ✅
   2. Duplicate csv_row_id: ⚠️


NameError: name 'mismatches' is not defined